# 電腦視覺技術與應用

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 說明電腦視覺的基本任務：影像分類、物件偵測、語意分割、實例分割與影像生成。
2. 使用 NumPy 表示灰階影像，理解像素矩陣、亮度、雜訊與影像前處理。
3. 以簡化方式示範邊緣偵測、特徵萃取與分類模型的關係。
4. 使用 scikit-learn 建立一個輕量影像分類流程，理解傳統機器學習與深度學習前的基本概念。
5. 以 TF-IDF 與相似度模擬多模態檢索概念，理解 CLIP 類模型背後的「共同語意空間」想法。

本練習不使用大型深度學習模型，而是用 Colab 預裝套件建立可執行、可觀察的輕量示範。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節會用到的 Python 套件，並建立固定亂數種子，讓每次執行結果一致。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
print('環境設定完成')


## 核心概念說明

電腦視覺的核心目標，是讓電腦能從影像或影片中擷取有意義的資訊，進而進行辨識、分析與決策。

常見任務包含：

| 任務 | 目的 | 範例 |
|---|---|---|
| 影像分類 | 判斷整張影像屬於哪一類 | 判斷圖片是貓、狗或商品類別 |
| 物件偵測 | 找出物件類別與位置 | 以邊界框標示行人與車輛 |
| 語意分割 | 將每個像素分類 | 分辨道路、天空、車輛、建築 |
| 實例分割 | 同時區分類別與不同個體 | 分辨畫面中每一台車 |
| 影像生成 | 產生或轉換影像 | 文字生成圖片、風格轉換 |

技術演進可簡化為三個階段：

1. **特徵工程時期**：人工設計邊緣、角點、紋理、顏色等特徵。
2. **深度學習時期**：CNN 自動從資料中學習影像特徵。
3. **多模態與生成式 AI 時期**：模型能結合文字與影像，進行跨模態理解與生成。

接下來會用簡化程式示範這些概念。


In [ ]:
# ── 示範：影像是像素矩陣 ──────────────────────────────
# 使用 NumPy 建立一張簡單灰階影像，觀察影像其實就是由數值矩陣組成。數值越大，像素越亮。

import numpy as np
import matplotlib.pyplot as plt

image = np.zeros((12, 12))
image[3:9, 3:9] = 0.7
image[5:7, 5:7] = 1.0

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap='gray', vmin=0, vmax=1)
plt.title('灰階影像：像素矩陣')
plt.colorbar(label='亮度')
plt.axis('off')
plt.show()

print('影像矩陣形狀:', image.shape)
print('最小亮度:', image.min())
print('最大亮度:', image.max())


In [ ]:
# ── 示範：影像前處理與邊緣偵測 ───────────────────────────
# 加入雜訊後，先用高斯濾波平滑影像，再用梯度近似邊緣偵測。這對應早期電腦視覺常見的影像處理與特徵工程流程。

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

np.random.seed(42)
image = np.zeros((64, 64))
image[18:46, 18:46] = 1.0
noise = np.random.normal(0, 0.25, image.shape)
noisy_image = np.clip(image + noise, 0, 1)

smoothed = ndimage.gaussian_filter(noisy_image, sigma=1.2)
gx = ndimage.sobel(smoothed, axis=1)
gy = ndimage.sobel(smoothed, axis=0)
edges = np.sqrt(gx ** 2 + gy ** 2)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
images = [image, noisy_image, smoothed, edges]
titles = ['原始影像', '加入雜訊', '平滑處理', '邊緣強度']

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('邊緣強度平均值:', round(float(edges.mean()), 4))


## 從特徵工程到影像分類

在深度學習普及前，常見流程是：

1. 將影像轉成可計算的數值。
2. 設計特徵，例如邊緣強度、亮度分布、紋理統計。
3. 將特徵輸入分類器，例如 SVM、KNN、Logistic Regression。

深度學習的 CNN 則把「特徵萃取」與「分類」整合在模型中，讓模型自動學習局部圖案、形狀與高階語意。

本練習使用 scikit-learn 的手寫數字資料集，建立一個輕量影像分類器，模擬影像分類任務的基本流程。


In [ ]:
# ── 實際應用：手寫數字影像分類 ───────────────────────────
# 使用 scikit-learn 內建的 digits 小型影像資料集，將 8x8 灰階影像攤平成特徵向量，訓練分類模型辨識數字。

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

X, y = load_digits(return_X_y=True)
images = X.reshape(-1, 8, 8)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print('測試資料筆數:', len(y_test))
print('分類準確率:', round(float(accuracy), 4))
print('混淆矩陣:')
print(confusion_matrix(y_test, y_pred))

sample_indices = np.arange(8)
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, idx in zip(axes, sample_indices):
    ax.imshow(X_test[idx].reshape(8, 8), cmap='gray')
    ax.set_title(f'預測:{y_pred[idx]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 跨模態語意檢索的輕量替代

影像分類讓模型判斷影像屬於哪一類；跨模態檢索則要讓「文字」與「影像」互相比對。真正的 CLIP 會把兩種模態投影到同一個語意空間，本練習改用影像描述文字與 TF-IDF 建立輕量替代示範，理解跨模態檢索的核心概念。


In [ ]:
# ── 示範：以 TF-IDF 模擬多模態語意檢索 ───────────────────
# 真正的 CLIP 會把影像與文字轉成同一語意空間中的向量。這裡用影像描述文字與 TF-IDF 建立輕量替代示範，理解跨模態檢索的核心概念。

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

image_catalog = pd.DataFrame({
    'image_id': ['img_001', 'img_002', 'img_003', 'img_004', 'img_005'],
    'caption': [
        '道路上有車輛與行人，適合自動駕駛場景理解',
        '醫療影像顯示肺部區域，可用於疾病輔助診斷',
        '工廠產線上的零件表面瑕疵檢測',
        '商店貨架上的商品分類與庫存辨識',
        '文字生成圖片與創意設計風格轉換'
    ]
})

query = '智慧製造 檢查 零件 瑕疵'
texts = image_catalog['caption'].tolist() + [query]

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
vectors = vectorizer.fit_transform(texts)
scores = cosine_similarity(vectors[-1], vectors[:-1]).ravel()

result = image_catalog.copy()
result['similarity'] = scores
result = result.sort_values('similarity', ascending=False)

print('查詢文字:', query)
print(result)
print('\n最相近的影像 ID:', result.iloc[0]['image_id'])


## 🧪 自我測驗

請完成下方 TODO 填空，實作簡化版影像分割：以亮度門檻把像素分成前景與背景，並計算前景比例。


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請完成下方 TODO 填空，實作簡化版影像分割：將亮度達到門檻（>=）的像素標為前景，並計算前景比例。

import numpy as np
import matplotlib.pyplot as plt

image = np.array([
    [0.1, 0.2, 0.2, 0.1, 0.0],
    [0.1, 0.8, 0.9, 0.7, 0.1],
    [0.2, 0.9, 1.0, 0.8, 0.2],
    [0.1, 0.7, 0.8, 0.6, 0.1],
    [0.0, 0.1, 0.2, 0.1, 0.0]
])

# TODO 1: 設定分割門檻，讓亮度大於等於 0.6 的像素成為前景
threshold = 0.6

# TODO 2: 產生前景遮罩，前景為 True，背景為 False
foreground_mask = image >= threshold

# TODO 3: 計算前景像素比例
foreground_ratio = foreground_mask.mean()

# Expected: threshold = 0.6
# Expected: foreground_ratio 約為 0.36（門檻用 >=，0.6 本身也算前景）
print('分割門檻:', threshold)
print('前景像素數:', int(foreground_mask.sum()))
print('全部像素數:', foreground_mask.size)
print('前景比例:', round(float(foreground_ratio), 2))

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('原始灰階影像')
axes[0].axis('off')

axes[1].imshow(foreground_mask, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('分割遮罩')
axes[1].axis('off')

plt.tight_layout()
plt.show()
